# XGBoostを初めて動かす

2つの測定値から合成スコアを予測する。両方が0.5以上の行だけスコアが加算される。1つの測定値だけでは見えない関係をモデルがどう近似するか、評価データで確かめる。

Apple Silicon Macの既存Docker/JupyterLabで、事前に[READMEのXGBoost手順](../README.md#最初のml実習--xgboost)を実施する。

1. **Run → Run All Cells** で学習と予測を実行する。比較相手は、すべての行に学習データの平均を返す基準モデル。2つのモデルを同じ評価用120行で比べる。MAE（平均絶対誤差）は、予測と正解の差の大きさを平均した値。
2. `max_depth = 3` を `1` に変え、学習・比較セルと描画セルを再実行する。変えるのは木の深さの上限だけ。MAEと、図で斜めの基準線から外れる点を比較する。
3. 結果をGitHubに残す。記録するのは深さごとのMAE、図、同じ評価データを使ったこと、観察した違い。この評価データは設定変更の比較にも使うため、設定選択後の独立な最終評価には別のデータが必要。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

print(f"XGBoost {xgb.__version__}")

# 2つの測定値から架空のスコアを作る。学習用と評価用で重複しない行を使う。
rng = np.random.default_rng(42)
measurements = rng.uniform(0, 1, size=(600, 2))
inputs = pd.DataFrame(measurements, columns=["measurement_a", "measurement_b"])
both_high = (inputs["measurement_a"] >= 0.5) & (inputs["measurement_b"] >= 0.5)
target = 40 + 10 * inputs["measurement_a"] + 35 * both_high
train_rows, test_rows = np.split(rng.permutation(len(inputs)), [480])

train_inputs = inputs.iloc[train_rows]
test_inputs = inputs.iloc[test_rows]
train_target = target.iloc[train_rows]
test_target = target.iloc[test_rows]

train = xgb.DMatrix(train_inputs, label=train_target)
test = xgb.DMatrix(test_inputs)


In [ ]:
max_depth = 3

model = xgb.train(
    {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "max_depth": max_depth,
        "eta": 0.2,
        "seed": 42,
        "nthread": 1,
    },
    train,
    num_boost_round=50,
)
prediction = model.predict(test)
baseline = np.full(len(test_inputs), train_target.mean())

comparison = pd.DataFrame({"actual": test_target.to_numpy(), "mean_baseline": baseline, "xgboost": prediction})
mae = comparison[["mean_baseline", "xgboost"]].sub(comparison["actual"], axis=0).abs().mean()
print(f"Assessment: {len(test_inputs)} held-out rows, max_depth={max_depth}")
print(f"Mean baseline MAE: {mae['mean_baseline']:.2f}")
print(f"XGBoost MAE: {mae['xgboost']:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(comparison["actual"], comparison["mean_baseline"], label="Training mean", alpha=0.6)
ax.scatter(comparison["actual"], comparison["xgboost"], label="XGBoost", alpha=0.6)
ax.plot([40, 85], [40, 85], "--", color="gray", label="Exact prediction")
ax.set(xlabel="Actual score", ylabel="Predicted score", title=f"Held-out predictions (max_depth={max_depth})")
ax.set_aspect("equal", adjustable="box")
ax.legend()
fig.tight_layout()
plt.show()
